In [2]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1) Load dataset (8x8 digit images)
X, y = load_digits(return_X_y=True)  # X shape: (n_samples, 64)

# 2) Train / Dev / Test split (exam concept: train/dev/test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# 3) Normalize/standardize features (important for distance-based methods)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_dev   = scaler.transform(X_dev)
X_test  = scaler.transform(X_test)
print("Feature means (train):", X_train.mean(axis=0))
print("Feature stds  (train):", X_train.std(axis=0))
# 4) Print dataset shapes
print(X_train.shape, X_dev.shape, X_test.shape)


Feature means (train): [ 0.00000000e+00 -1.73639705e-16 -2.71891353e-17  1.37593503e-16
  3.17206578e-17 -8.02285470e-17  1.63855736e-16  1.51033587e-16
 -1.23767210e-16 -6.47801746e-17 -1.19673391e-16 -1.36357633e-16
  6.72519142e-17 -1.68490248e-16  5.68500102e-17 -5.16593571e-16
 -2.94548966e-16  3.15146795e-17 -6.13815327e-17  1.09271487e-16
  4.20195727e-17  6.79728382e-17  4.99497372e-17 -6.02306290e-16
  1.49379324e-16 -2.13393516e-16 -7.87866989e-18 -7.78597965e-17
  8.34212106e-18 -5.93217497e-17 -3.56342455e-17 -5.45456280e-17
  0.00000000e+00 -8.18763733e-17 -1.68902204e-17 -3.75910393e-18
 -6.00426738e-17  9.71187674e-17 -2.81984290e-16  0.00000000e+00
 -1.29122645e-16  3.95478332e-17 -5.95277280e-17  1.70961987e-17
 -9.31021906e-17  1.67202883e-16 -7.49761004e-17 -1.89500034e-16
 -1.20883513e-16  5.50477001e-17  7.55940353e-17 -3.08967447e-18
 -1.80436989e-16 -1.37207294e-16  1.32856002e-17 -1.78995141e-16
  0.00000000e+00 -1.05975834e-16  1.11640237e-16  7.95076229e-17
  

In [3]:
def pairwise_distances(XA, XB, metric="euclidean"):
    """
    XA: (nA, d), XB: (nB, d)
    returns: (nA, nB) distances
    """
    if metric == "euclidean":
        # ||a-b||^2 = ||a||^2 + ||b||^2 - 2 a.b
        A2 = np.sum(XA**2, axis=1, keepdims=True)      # (nA, 1)
        B2 = np.sum(XB**2, axis=1, keepdims=True).T    # (1, nB)
        D2 = A2 + B2 - 2 * XA @ XB.T
        D2 = np.maximum(D2, 0)  # numerical safety
        return np.sqrt(D2)

    if metric == "manhattan":
        # broadcasting: (nA, 1, d) - (1, nB, d) -> (nA, nB, d)
        return np.sum(np.abs(XA[:, None, :] - XB[None, :, :]), axis=2)

    if metric == "cosine":
        # cosine distance = 1 - cosine similarity
        XA_norm = XA / (np.linalg.norm(XA, axis=1, keepdims=True) + 1e-12)
        XB_norm = XB / (np.linalg.norm(XB, axis=1, keepdims=True) + 1e-12)
        sim = XA_norm @ XB_norm.T
        return 1 - sim

    raise ValueError("metric must be euclidean, manhattan, or cosine")


def knn_predict(X_train, y_train, X_query, k=5, metric="euclidean"):
    D = pairwise_distances(X_query, X_train, metric=metric)  # (n_query, n_train)
    nn_idx = np.argpartition(D, kth=k-1, axis=1)[:, :k]      # (n_query, k) fast top-k
    nn_labels = y_train[nn_idx]                               # (n_query, k)

    # majority vote (works for multiclass)
    preds = []
    for row in nn_labels:
        counts = np.bincount(row, minlength=np.max(y_train)+1)
        preds.append(np.argmax(counts))
    return np.array(preds)


In [5]:
def eval_knn_over_k(k_values, metric="euclidean"):
    results = []
    for k in k_values:
        y_pred = knn_predict(X_train, y_train, X_dev, k=k, metric=metric)
        acc = accuracy_score(y_dev, y_pred)
        results.append((k, acc))
    return results

k_values = list(range(1, 21, 2))  # odd k avoids ties (also in your slides) :contentReference[oaicite:10]{index=10}

for metric in ["euclidean", "manhattan", "cosine"]:
    res = eval_knn_over_k(k_values, metric=metric)
    best_k, best_acc = max(res, key=lambda x: x[1])
    print(metric, "best_k=", best_k, "dev_acc=", round(best_acc, 4))


euclidean best_k= 1 dev_acc= 0.9721
manhattan best_k= 5 dev_acc= 0.9721
cosine best_k= 1 dev_acc= 0.9721
